In [ ]:
%sql
--DROP SCHEMA IF EXISTS cnpj_data_lakehouse.default CASCADE;
--DROP SCHEMA IF EXISTS cnpj_data_lakehouse.bronze CASCADE;
--DROP SCHEMA IF EXISTS cnpj_data_lakehouse.silver CASCADE;
--DROP SCHEMA IF EXISTS cnpj_data_lakehouse.gold CASCADE;
--CREATE CATALOG IF NOT EXISTS cnpj_data_lakehouse;
--CREATE SCHEMA IF NOT EXISTS cnpj_data_lakehouse.bronze;
--CREATE VOLUME IF NOT EXISTS cnpj_data_lakehouse.bronze.raw;
--CREATE SCHEMA IF NOT EXISTS cnpj_data_lakehouse.silver;
--CREATE VOLUME IF NOT EXISTS cnpj_data_lakehouse.silver.raw;
--CREATE VOLUME IF NOT EXISTS cnpj_data_lakehouse.silver.cleaned;
--CREATE SCHEMA IF NOT EXISTS cnpj_data_lakehouse.gold;
--CREATE VOLUME IF NOT EXISTS cnpj_data_lakehouse.gold.bridge;
--CREATE VOLUME IF NOT EXISTS cnpj_data_lakehouse.gold.dim;
--CREATE VOLUME IF NOT EXISTS cnpj_data_lakehouse.gold.fact;
--CREATE VOLUME IF NOT EXISTS cnpj_data_lakehouse.gold.int;
--CREATE VOLUME IF NOT EXISTS cnpj_data_lakehouse.gold.agg;

In [0]:
import os
import datetime as dt

ano_mes = dt.datetime.now().strftime("%Y_%m")
volumes = os.listdir('/Volumes/cnpj_data_lakehouse/gold') #bridge, dim, fact, int, agg

for volume in volumes:
  if(volume.startswith('int')): continue 
      
  files = os.listdir(f"/Volumes/cnpj_data_lakehouse/gold/{volume}/{ano_mes}")       
      
  for file in files:
    table_name = file.replace('.parquet','')        

    spark.sql(f"DROP TABLE IF EXISTS cnpj_data_lakehouse.gold.{table_name}")    
    
    spark.sql(f"CREATE OR REPLACE TABLE cnpj_data_lakehouse.gold.{table_name} USING DELTA AS SELECT * FROM parquet.`/Volumes/cnpj_data_lakehouse/gold/{volume}/{ano_mes}/{file}`")
    
    if(table_name.startswith('agg_')): continue

    sql_alter_table = f"ALTER TABLE cnpj_data_lakehouse.gold.{table_name}"

    match table_name:
        case 'bridge_estabelecimentos_socios':
            spark.sql(f"{sql_alter_table} ALTER COLUMN sk_estabelecimento SET NOT NULL")
            spark.sql(f"{sql_alter_table} ALTER COLUMN sk_socio SET NOT NULL")
            spark.sql(f"{sql_alter_table} ADD CONSTRAINT pk_{table_name}_sk_id PRIMARY KEY (sk_estabelecimento, sk_socio)")

        case 'bridge_estabelecimentos_cnaes':
            spark.sql(f"{sql_alter_table} ALTER COLUMN sk_estabelecimento SET NOT NULL")
            spark.sql(f"{sql_alter_table} ALTER COLUMN sk_cnae SET NOT NULL")
            spark.sql(f"{sql_alter_table} ADD CONSTRAINT pk_{table_name}_sk_id PRIMARY KEY (sk_estabelecimento, sk_cnae)")
        
        case 'fact_estabelecimentos':
            spark.sql(f"{sql_alter_table} ALTER COLUMN sk_estabelecimento SET NOT NULL")
            spark.sql(f"{sql_alter_table} ADD CONSTRAINT pk_{table_name}_sk_estabelecimento PRIMARY KEY (sk_estabelecimento)")        

        case _:    
            spark.sql(f"{sql_alter_table} ALTER COLUMN sk_id SET NOT NULL")
            spark.sql(f"{sql_alter_table} ADD CONSTRAINT pk_{table_name}_sk_id PRIMARY KEY (sk_id)")

In [0]:
%sql
--agg_fact_estabelecimentos_cnaes
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos_cnaes DROP CONSTRAINT IF EXISTS fk_agg_fact_estabelecimentos_cnaes_dim_cnaes; 
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos_cnaes ADD CONSTRAINT fk_agg_fact_estabelecimentos_cnaes_dim_cnaes FOREIGN KEY (sk_cnae) REFERENCES cnpj_data_lakehouse.gold.dim_cnaes(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos_cnaes DROP CONSTRAINT IF EXISTS fk_agg_fact_estabelecimentos_cnaes_dim_tempo_inicio_atividades; 
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos_cnaes ADD CONSTRAINT fk_agg_fact_estabelecimentos_cnaes_dim_tempo_inicio_atividades FOREIGN KEY (sk_tempo_inicio_atividade) REFERENCES cnpj_data_lakehouse.gold.dim_tempo_inicio_atividades(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos_cnaes DROP CONSTRAINT IF EXISTS fk_agg_fact_estabelecimentos_cnaes_dim_situacoes; 
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos_cnaes ADD CONSTRAINT fk_agg_fact_estabelecimentos_cnaes_dim_situacoes FOREIGN KEY (sk_situacoes) REFERENCES cnpj_data_lakehouse.gold.dim_situacoes(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos_cnaes DROP CONSTRAINT IF EXISTS fk_agg_fact_estabelecimentos_cnaes_dim_localidades; 
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos_cnaes ADD CONSTRAINT fk_agg_fact_estabelecimentos_cnaes_dim_localidades FOREIGN KEY (sk_localidades) REFERENCES cnpj_data_lakehouse.gold.dim_localidades(sk_id);

--agg_fact_estabelecimentos
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos DROP CONSTRAINT IF EXISTS fk_agg_fact_estabelecimentos_dim_tempo_inicio_atividades; 
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos ADD CONSTRAINT fk_agg_fact_estabelecimentos_dim_tempo_inicio_atividades FOREIGN KEY (sk_tempo_inicio_atividade) REFERENCES cnpj_data_lakehouse.gold.dim_tempo_inicio_atividades(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos DROP CONSTRAINT IF EXISTS fk_agg_fact_estabelecimentos_dim_situacoes; 
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos ADD CONSTRAINT fk_agg_fact_estabelecimentos_dim_situacoes FOREIGN KEY (sk_situacoes) REFERENCES cnpj_data_lakehouse.gold.dim_situacoes(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos DROP CONSTRAINT IF EXISTS fk_agg_fact_estabelecimentos_dim_localidades; 
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos ADD CONSTRAINT fk_agg_fact_estabelecimentos_dim_localidades FOREIGN KEY (sk_localidades) REFERENCES cnpj_data_lakehouse.gold.dim_localidades(sk_id);

--bridge_estabelecimentos_cnaes
ALTER TABLE cnpj_data_lakehouse.gold.bridge_estabelecimentos_cnaes DROP CONSTRAINT IF EXISTS fk_bridge_estabelecimentos_cnaes_fact_estabelecimentos; 
ALTER TABLE cnpj_data_lakehouse.gold.bridge_estabelecimentos_cnaes ADD CONSTRAINT fk_bridge_estabelecimentos_cnaes_fact_estabelecimentos FOREIGN KEY (sk_estabelecimento) REFERENCES cnpj_data_lakehouse.gold.fact_estabelecimentos(sk_estabelecimento);
ALTER TABLE cnpj_data_lakehouse.gold.bridge_estabelecimentos_cnaes DROP CONSTRAINT IF EXISTS fk_bridge_estabelecimentos_cnaes_dim_cnaes; 
ALTER TABLE cnpj_data_lakehouse.gold.bridge_estabelecimentos_cnaes ADD CONSTRAINT fk_bridge_estabelecimentos_cnaes_dim_cnaes FOREIGN KEY (sk_cnae) REFERENCES cnpj_data_lakehouse.gold.dim_cnaes(sk_id);

--bridge_estabelecimentos_socios
ALTER TABLE cnpj_data_lakehouse.gold.bridge_estabelecimentos_socios DROP CONSTRAINT IF EXISTS fk_bridge_estabelecimentos_socios_fact_estabelecimentos; 
ALTER TABLE cnpj_data_lakehouse.gold.bridge_estabelecimentos_socios ADD CONSTRAINT fk_bridge_estabelecimentos_socios_fact_estabelecimentos FOREIGN KEY (sk_estabelecimento) REFERENCES cnpj_data_lakehouse.gold.fact_estabelecimentos(sk_estabelecimento);
ALTER TABLE cnpj_data_lakehouse.gold.bridge_estabelecimentos_socios DROP CONSTRAINT IF EXISTS fk_bridge_estabelecimentos_socios_dim_socios;
ALTER TABLE cnpj_data_lakehouse.gold.bridge_estabelecimentos_socios ADD CONSTRAINT fk_bridge_estabelecimentos_socios_dim_socios FOREIGN KEY (sk_socio) REFERENCES cnpj_data_lakehouse.gold.dim_socios(sk_id);

--fact_estabelecimentos
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos DROP CONSTRAINT IF EXISTS fk_fact_estabelecimentos_dim_tempo_inicio_atividades; 
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_tempo_inicio_atividades FOREIGN KEY (sk_tempo_inicio_atividade) REFERENCES cnpj_data_lakehouse.gold.dim_tempo_inicio_atividades(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos DROP CONSTRAINT IF EXISTS fk_fact_estabelecimentos_dim_tempo_situacao_cadastral; 
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_tempo_situacao_cadastral FOREIGN KEY (sk_tempo_situacoes_cadastrais) REFERENCES cnpj_data_lakehouse.gold.dim_tempo_situacoes_cadastrais(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos DROP CONSTRAINT IF EXISTS fk_fact_estabelecimentos_dim_tempo_situacao_especial; 
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_tempo_situacao_especial FOREIGN KEY (sk_tempo_situacoes_especiais) REFERENCES cnpj_data_lakehouse.gold.dim_tempo_situacoes_especiais(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos DROP CONSTRAINT IF EXISTS fk_fact_estabelecimentos_dim_situacoes;
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_situacoes FOREIGN KEY (sk_situacoes) REFERENCES cnpj_data_lakehouse.gold.dim_situacoes(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos DROP CONSTRAINT IF EXISTS fk_fact_estabelecimentos_dim_localidades; 
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_localidades FOREIGN KEY (sk_localidades) REFERENCES cnpj_data_lakehouse.gold.dim_localidades(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos DROP CONSTRAINT IF EXISTS fk_fact_estabelecimentos_dim_naturezas_juridicas; 
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_naturezas_juridicas FOREIGN KEY (sk_naturezas_juridicas) REFERENCES cnpj_data_lakehouse.gold.dim_naturezas_juridicas(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos DROP CONSTRAINT IF EXISTS fk_fact_estabelecimentos_dim_cadastro; 
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_cadastro FOREIGN KEY (sk_estabelecimento) REFERENCES cnpj_data_lakehouse.gold.dim_cadastro(sk_id);

-- Ativando o Liquid Clustering nas tabelas
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos_cnaes CLUSTER BY (sk_cnae, sk_localidades, sk_tempo_inicio_atividade, sk_situacoes);
ALTER TABLE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos CLUSTER BY (sk_tempo_inicio_atividade, sk_situacoes, sk_localidades);
ALTER TABLE cnpj_data_lakehouse.gold.bridge_estabelecimentos_cnaes CLUSTER BY (sk_estabelecimento, sk_cnae);
ALTER TABLE cnpj_data_lakehouse.gold.bridge_estabelecimentos_socios CLUSTER BY (sk_estabelecimento, sk_socio);
ALTER TABLE cnpj_data_lakehouse.gold.dim_cadastro CLUSTER BY (cnpj_completo);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos CLUSTER BY (sk_tempo_inicio_atividade, sk_situacoes, sk_localidades, sk_naturezas_juridicas);
OPTIMIZE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos_cnaes FULL;
OPTIMIZE cnpj_data_lakehouse.gold.agg_fact_estabelecimentos FULL;
OPTIMIZE cnpj_data_lakehouse.gold.bridge_estabelecimentos_cnaes FULL;
OPTIMIZE cnpj_data_lakehouse.gold.bridge_estabelecimentos_socios FULL;
OPTIMIZE cnpj_data_lakehouse.gold.dim_cadastro FULL;
OPTIMIZE cnpj_data_lakehouse.gold.fact_estabelecimentos FULL;